# Gold Layer — Business-Ready Data

This notebook prepares the cleaned vehicle listings for analysis.

The Silver data will be organized into one fact table and three dimension tables to support vehicle price and demand analysis.

**Source:** `car_sales.silver.vehicles_clean`

**Output tables**
- `fact_listings`
- `dim_vehicle`
- `dim_location`
- `dim_date`

In [0]:
from pyspark.sql import functions as F

df_silver = spark.table("car_sales.silver.vehicles_clean")

print(f"Silver rows: {df_silver.count():,}")
print(f"Available columns: {len(df_silver.columns)}")

Silver rows: 392,812
Available columns: 28


## 1. Create the Listings Fact Table

The fact table stores the main details used to analyze vehicle listings.

It includes the listing ID, price, model year, odometer reading, and the vehicle, location, and posting date fields needed for analysis.

Each row represents one listing from the Silver dataset.

In [0]:
df_fact_listings = df_silver.select(
    "id",
    "price",
    "year",
    "odometer",
    "manufacturer",
    "model",
    "region",
    "state",
    "lat",
    "long",
    "posting_date",
    "fuel",
    "transmission",
    "condition",
    "type",
    "title_status"
)

print(f"Fact table rows: {df_fact_listings.count():,}")
display(df_fact_listings.limit(10))

Fact table rows: 392,812


id,price,year,odometer,manufacturer,model,region,state,lat,long,posting_date,fuel,transmission,condition,type,title_status
7308745660,35590.0,2011,16796.0,dodge,challenger srt8 coupe 2d,zanesville / cambridge,oh,39.95,-82.01,2021-04-18T11:20:49-0400,gas,other,good,coupe,clean
7308706421,21590.0,2011,23483.0,mazda,mx-5 miata grand touring,zanesville / cambridge,oh,39.95,-82.01,2021-04-18T10:00:50-0400,gas,other,good,convertible,clean
7308323514,38990.0,2020,9050.0,acura,rdx sh-awd technology pkg,zanesville / cambridge,oh,39.95,-82.01,2021-04-17T13:10:45-0400,gas,other,good,SUV,clean
7308249624,39990.0,2020,30327.0,acura,rdx sh-awd advance pkg,zanesville / cambridge,oh,39.95,-82.01,2021-04-17T11:21:19-0400,gas,other,good,SUV,clean
7308249496,47590.0,2017,63344.0,mercedes-benz,gls 550 sport,zanesville / cambridge,oh,39.95,-82.01,2021-04-17T11:21:09-0400,other,other,good,SUV,clean
7308191882,40990.0,2018,54431.0,mercedes-benz,gls 450 4matic,zanesville / cambridge,oh,39.95,-82.01,2021-04-17T09:30:41-0400,gas,automatic,good,SUV,clean
7308192431,9795.0,1999,74000.0,dodge,durango,zanesville / cambridge,oh,39.998337,-82.022885,2021-04-17T09:31:49-0400,gas,automatic,like new,SUV,clean
7307705211,22931.0,2018,22514.0,bmw,320 xi,zanesville / cambridge,oh,39.884989,-82.967588,2021-04-16T11:02:20-0400,gas,automatic,excellent,sedan,null
7307668799,19786.0,2016,56536.0,bmw,528 i,zanesville / cambridge,oh,39.884989,-82.967588,2021-04-16T10:02:31-0400,gas,automatic,excellent,sedan,null
7307556582,2595.0,2007,206000.0,nissan,quest,zanesville / cambridge,oh,39.766768,-82.091297,2021-04-15T23:36:24-0400,gas,automatic,excellent,van,clean


## 2. Save the Listings Fact Table

We will save the prepared listings data as a Delta table in the Gold schema.

This table will serve as the main source for vehicle price and listing analysis.

In [0]:
(
    df_fact_listings.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("car_sales.gold.fact_listings")
)

print("Fact table saved successfully.")

Fact table saved successfully.


## 3. Create the Vehicle Dimension

The vehicle dimension stores descriptive information about each vehicle.

We will keep distinct combinations of vehicle details so that listings can be analyzed by manufacturer, model, model year, fuel type, transmission, condition, and vehicle type.

In [0]:
df_dim_vehicle = (
    df_silver
    .select(
        "manufacturer",
        "model",
        "year",
        "fuel",
        "transmission",
        "condition",
        "cylinders",
        "drive",
        "size",
        "type"
    )
    .dropDuplicates()
)

print(f"Vehicle dimension rows: {df_dim_vehicle.count():,}")
display(df_dim_vehicle.limit(10))

Vehicle dimension rows: 170,238


manufacturer,model,year,fuel,transmission,condition,cylinders,drive,size,type
kia,rio,2008,gas,automatic,null,4,fwd,null,sedan
ford,transit 150,2017,gas,automatic,excellent,6,rwd,full-size,van
bmw,x5,2008,gas,automatic,null,null,null,null,null
chevrolet,silverado 1500 lt,2016,gas,automatic,good,6,rwd,full-size,pickup
jaguar,xjr s,2004,gas,automatic,good,8,rwd,full-size,sedan
chevrolet,silverado 3500,2015,diesel,automatic,null,null,null,null,null
infiniti,m37,2011,gas,automatic,excellent,6,null,null,null
gmc,sierra 1500,1994,gas,automatic,null,6,null,null,null
toyota,sienna,2011,gas,automatic,null,6,fwd,null,mini-van
chevrolet,lumina,1998,gas,automatic,fair,null,null,null,null


## 4. Save the Vehicle Dimension

We will save the distinct vehicle details as a Delta table in the Gold schema.

This table will support analysis by manufacturer, model, year, and other vehicle characteristics.

In [0]:
(
    df_dim_vehicle.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("car_sales.gold.dim_vehicle")
)

print("Vehicle dimension saved successfully.")

Vehicle dimension saved successfully.


## 5. Create the Location Dimension

The location dimension stores the geographic details associated with vehicle listings.

We will keep distinct combinations of region, state, latitude, and longitude. These fields will support analysis of vehicle prices and listing distribution by location.

In [0]:
df_dim_location = (
    df_silver
    .select(
        "region",
        "state",
        "lat",
        "long"
    )
    .dropDuplicates()
)

print(f"Location dimension rows: {df_dim_location.count():,}")
display(df_dim_location.limit(10))

Location dimension rows: 70,179


region,state,lat,long
birmingham,al,34.982021,-88.465261
huntsville / decatur,al,34.559,-85.7921
mobile,al,30.5808,-88.2298
dothan,al,26.0218,-80.1891
birmingham,al,33.5409,-86.7495
birmingham,al,33.345455,-86.624794
dothan,al,29.801374,-82.531052
dothan,al,30.870102,-85.162839
birmingham,al,33.4816,-86.859
birmingham,al,32.332046,-90.857163


## 6. Save the Location Dimension

We will save the distinct location records as a Delta table in the Gold schema.

This table will support analysis of listing distribution and vehicle prices across regions and states.

In [0]:
(
    df_dim_location.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("car_sales.gold.dim_location")
)

print("Location dimension saved successfully.")

Location dimension saved successfully.


## 7. Create the Date Dimension

The date dimension stores calendar details for each vehicle listing date.

We will extract the date from `posting_date` and create separate columns for the year, month, day, and day of the week. These fields will support analysis of listing activity over time.

In [0]:
df_dim_date = (
    df_silver
    .withColumn(
        "listing_date",
        F.to_date(F.to_timestamp("posting_date"))
    )
    .filter(F.col("listing_date").isNotNull())
    .select(
        "listing_date",
        F.year("listing_date").alias("year"),
        F.month("listing_date").alias("month"),
        F.dayofmonth("listing_date").alias("day"),
        F.date_format("listing_date", "EEEE").alias("day_of_week")
    )
    .dropDuplicates()
)

print(f"Date dimension rows: {df_dim_date.count():,}")
display(df_dim_date.orderBy("listing_date").limit(10))

Date dimension rows: 32


listing_date,year,month,day,day_of_week
2021-04-04,2021,4,4,Sunday
2021-04-05,2021,4,5,Monday
2021-04-06,2021,4,6,Tuesday
2021-04-07,2021,4,7,Wednesday
2021-04-08,2021,4,8,Thursday
2021-04-09,2021,4,9,Friday
2021-04-10,2021,4,10,Saturday
2021-04-11,2021,4,11,Sunday
2021-04-12,2021,4,12,Monday
2021-04-13,2021,4,13,Tuesday


## 8. Save the Date Dimension

We will save the distinct listing dates and their calendar details as a Delta table in the Gold schema.

This table will support analysis of listing activity by date, month, and day of the week.

In [0]:
(
    df_dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("car_sales.gold.dim_date")
)

print("Date dimension saved successfully.")

Date dimension saved successfully.


## 9. Verify the Gold Tables

We will check that all four Gold tables were saved successfully and contain data.

This final check confirms that the tables are available for reporting and analysis.

In [0]:
gold_tables = [
    "fact_listings",
    "dim_vehicle",
    "dim_location",
    "dim_date"
]

for table_name in gold_tables:
    table = spark.table(f"car_sales.gold.{table_name}")
    print(f"{table_name}: {table.count():,} rows")

fact_listings: 392,812 rows
dim_vehicle: 170,238 rows
dim_location: 70,179 rows
dim_date: 32 rows


## 10. Review Vehicle Prices

Before preparing the data for reporting, we will review the price distribution to identify unusually high values.

This check will help us choose a reasonable price range for analysis without changing the original Silver data.

In [0]:
display(
    df_silver.select(
        F.min("price").alias("minimum_price"),
        F.expr("percentile_approx(price, 0.50)").alias("median_price"),
        F.expr("percentile_approx(price, 0.95)").alias("95th_percentile_price"),
        F.expr("percentile_approx(price, 0.99)").alias("99th_percentile_price"),
        F.max("price").alias("maximum_price")
    )
)

minimum_price,median_price,95th_percentile_price,99th_percentile_price,maximum_price
1.0,15000.0,44998.0,67999.0,3.736928711E9


## 11. Review High-Price Listings

The price distribution shows that 99% of listings are priced at approximately $68,000 or below.

We will examine listings above $100,000 before deciding whether to exclude them from reporting.

The original Silver data will remain unchanged.

In [0]:
price_ranges = df_silver.select(
    F.count("*").alias("total_listings"),
    F.count(F.when(F.col("price") > 100000, 1)).alias("above_100k"),
    F.count(F.when(F.col("price") > 200000, 1)).alias("above_200k"),
    F.count(F.when(F.col("price") > 1000000, 1)).alias("above_1m")
)

display(price_ranges)

total_listings,above_100k,above_200k,above_1m
392812,654,124,53


## 12. Prepare Data for Price Analysis

Some listings have unusually high prices that can distort average-price comparisons.

For the Gold reporting table, we will keep listings priced at $100,000 or below. The original Silver table remains unchanged, so the excluded listings are still available for other analysis.

In [0]:
df_fact_listings = (
    df_silver
    .filter(F.col("price") <= 100000)
    .select(
        "id",
        "price",
        "year",
        "odometer",
        "manufacturer",
        "model",
        "region",
        "state",
        "lat",
        "long",
        "posting_date",
        "fuel",
        "transmission",
        "condition",
        "type",
        "title_status"
    )
)

print(f"Listings prepared for reporting: {df_fact_listings.count():,}")
display(df_fact_listings.limit(10))

Listings prepared for reporting: 392,158


id,price,year,odometer,manufacturer,model,region,state,lat,long,posting_date,fuel,transmission,condition,type,title_status
7316814884,33590.0,2014,57923.0,gmc,sierra 1500 crew cab slt,auburn,al,32.59,-85.48,2021-05-04T12:31:18-0500,gas,other,good,pickup,clean
7316814758,22590.0,2010,71229.0,chevrolet,silverado 1500,auburn,al,32.59,-85.48,2021-05-04T12:31:08-0500,gas,other,good,pickup,clean
7316814989,39590.0,2020,19160.0,chevrolet,silverado 1500 crew,auburn,al,32.59,-85.48,2021-05-04T12:31:25-0500,gas,other,good,pickup,clean
7316743432,30990.0,2017,41124.0,toyota,tundra double cab sr,auburn,al,32.59,-85.48,2021-05-04T10:41:31-0500,gas,other,good,pickup,clean
7316356412,15000.0,2013,128000.0,ford,f-150 xlt,auburn,al,32.592,-85.5189,2021-05-03T14:02:03-0500,gas,automatic,excellent,truck,clean
7316343444,27990.0,2012,68696.0,gmc,sierra 2500 hd extended cab,auburn,al,32.59,-85.48,2021-05-03T13:41:25-0500,gas,other,good,pickup,clean
7316304717,34590.0,2016,29499.0,chevrolet,silverado 1500 double,auburn,al,32.59,-85.48,2021-05-03T12:41:33-0500,gas,other,good,pickup,clean
7316285779,35000.0,2019,43000.0,toyota,tacoma,auburn,al,32.6013,-85.443974,2021-05-03T12:12:59-0500,gas,automatic,excellent,truck,clean
7316257769,29990.0,2016,17302.0,chevrolet,colorado extended cab,auburn,al,32.59,-85.48,2021-05-03T11:31:14-0500,gas,other,good,pickup,clean
7316133914,38590.0,2011,30237.0,chevrolet,corvette grand sport,auburn,al,32.59,-85.48,2021-05-03T07:50:35-0500,gas,other,good,other,clean


### Save the Reporting Fact Table

Save the filtered listings to the Gold table used for price and demand analysis.

In [0]:
(
    df_fact_listings.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("car_sales.gold.fact_listings")
)

print("Gold fact table updated successfully.")

Gold fact table updated successfully.


## 13. Verify the Reporting Fact Table

We will confirm that the saved Gold fact table contains the expected number of listings after applying the price filter.

In [0]:
df_fact_saved = spark.table("car_sales.gold.fact_listings")

print(f"Final reporting listings: {df_fact_saved.count():,}")
print(f"Maximum reporting price: ${df_fact_saved.agg(F.max('price')).first()[0]:,.2f}")

Final reporting listings: 392,158
Maximum reporting price: $100,000.00


## 14. Project Summary

The Gold layer is ready for reporting and analysis.

The project now contains a cleaned Silver table and four Gold tables covering vehicle listings, vehicle details, locations, and listing dates.

The reporting fact table excludes listings priced above $100,000 to reduce the effect of extreme prices on price comparisons.

**Source:** `car_sales.silver.vehicles_clean`

**Gold tables**
- `car_sales.gold.fact_listings`
- `car_sales.gold.dim_vehicle`
- `car_sales.gold.dim_location`
- `car_sales.gold.dim_date`

**Final reporting listings:** 392,158